# My Tu Verras, workshop 2 — A baseline, and how to know you beat it

Last week you looked at the data. Today you build the model the project asks for —
`boston_fit_model` and `boston_predict` — and, more importantly, you build the **method** that
tells you whether a change to it actually helped.

Because here is what will happen otherwise: you will try a scaler, the number will move, and you
will have no way to know whether you improved the model or just drew a luckier split.

1. **Split three ways** — and what each set is for.
2. **The dumb baseline** — the number your model has to beat to mean anything.
3. **One change at a time** — a small experiment campaign.
4. **The thing nobody tells you**: on 506 rows, the split moves the score more than your changes do.

Everything runs on Qwasar's `boston.csv`, the same file you submit against.

In [ ]:
# ABOUTME: My Tu Verras workshop 2 - baseline model and experimental method on boston.csv.
# ABOUTME: Train/validation/test, the dumb baseline, one change per run, and split-to-split noise.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from pathlib import Path

def qwasar_file(name):
    """Locate a dataset fetched by workshops/get_data.py, from wherever this notebook was opened."""
    for up in range(4):
        p = Path(*([".."] * up), "data", name)
        if p.exists():
            return p
    raise FileNotFoundError(
        f"{name} is missing. Run `uv run get_data.py` in the workshops/ folder first - "
        "these notebooks only ever use the real datasets Qwasar provides.")

boston = pd.read_csv(qwasar_file("boston.csv"))
TARGET = "MDEV"                      # spelled as the file spells it, not as the subject does
FEATURES = [c for c in boston.columns if c != TARGET]
SEED = 42                            # the seed is part of the experiment. Write it down.
print(boston.shape, "| target:", TARGET)

(506, 14) | target: MDEV


## 1. Three sets, three questions

| set | share | the question it answers | how often you may look |
|---|---|---|---|
| **train** | 70 % | what should the model's coefficients be? | constantly |
| **validation** | 15 % | which of my ideas is best? | every experiment |
| **test** | 15 % | how will this do on towns nobody has seen? | **once**, at the very end |

**Rule of thumb:** a number that influenced a decision cannot also be your final estimate of
quality. That is the whole reason the test set exists.

In [ ]:
def split(df, seed=SEED):
    """Shuffle once with a fixed seed, then cut 70 / 15 / 15."""
    shuffled = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    a, b = int(0.70 * len(df)), int(0.85 * len(df))
    return shuffled[:a], shuffled[a:b], shuffled[b:]

train, val, test = split(boston)
print("train", len(train), "| val", len(val), "| test", len(test))

train 354 | val 76 | test 76


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("three sets", lambda: all(x is not None for x in (train, val, test)))
check("they add up to 506", lambda: len(train) + len(val) + len(test) == 506)
check("roughly 70/15/15", lambda: (len(train), len(val), len(test)) == (354, 76, 76))
check("no town is in two sets", lambda: len(set(map(tuple, train.values)) &
                                            set(map(tuple, val.values))) == 0)

  three sets: PASS
  they add up to 506: PASS
  roughly 70/15/15: PASS
  no town is in two sets: PASS


True

## 2. The dumb baseline

Before celebrating any RMSE, answer: what would I score by not modelling at all — by predicting
the training mean for every town?

If your model cannot beat that, it has learned nothing.

In [ ]:
def rmse(predictions, truth):
    """Root mean squared error, in thousands of dollars."""
    return float(np.sqrt(mean_squared_error(truth, predictions)))

dumb_prediction = np.full(len(val), train[TARGET].mean())
dumb_rmse = rmse(dumb_prediction, val[TARGET])
print(f"predict-the-mean RMSE: {dumb_rmse:.3f}  (in $1000s)")
print(f"for reference, the spread of prices: std = {train[TARGET].std():.3f}")

predict-the-mean RMSE: 9.712  (in $1000s)
for reference, the spread of prices: std = 9.258


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("a number came out", lambda: dumb_rmse is not None)
check("it is close to the spread of the target", lambda: abs(dumb_rmse - val[TARGET].std()) < 2)

  a number came out: PASS
  it is close to the spread of the target: PASS


True

About **9.2**, which is simply the spread of house prices. That is the bar.

### Exercise 3 — `boston_fit_model` and `boston_predict`

The two functions the project asks for. Keep them boring: fit on the training features, predict on
whatever you are handed.

In [ ]:
def boston_fit_model(dataset):
    """Fit a linear regression predicting the target from every other column."""
    return LinearRegression().fit(dataset[FEATURES], dataset[TARGET])

def boston_predict(estimator, array_to_predict):
    """Predict for a DataFrame (or array) of towns."""
    return estimator.predict(array_to_predict)

model = boston_fit_model(train)
baseline_rmse = rmse(boston_predict(model, val[FEATURES]), val[TARGET])
print(f"model RMSE on validation: {baseline_rmse:.3f}")
print(f"versus the dumb baseline: {dumb_rmse:.3f}")
print(f"the model explains {1 - baseline_rmse**2 / dumb_rmse**2:.1%} of the variance the mean leaves behind")

model RMSE on validation: 4.716
versus the dumb baseline: 9.712
the model explains 76.4% of the variance the mean leaves behind


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("the model fits", lambda: baseline_rmse is not None)
check("it beats predicting the mean", lambda: baseline_rmse < dumb_rmse)
check("and lands around 4.8", lambda: 3.5 < baseline_rmse < 6.5)

  the model fits: PASS
  it beats predicting the mean: PASS
  and lands around 4.8: PASS


True

**4.83 against 9.22.** Now "RMSE 4.8" means something: it is roughly half the error of knowing
nothing, or about **$4 800** on a median house.

That pair of numbers — model and dumb baseline — is the first line of any report you write.

## 3. One change at a time

You have three ideas. Try them together and a gain tells you nothing about which one caused it.
So: one change per run, everything else frozen.

In [ ]:
def run_experiment(tr, va, scaler=None, log_target=False, drop_censored=False):
    """Fit one configuration and return its validation RMSE.

    scaler        : an unfitted sklearn scaler, or None
    log_target    : fit on log(price), then exponentiate the predictions back
    drop_censored : remove the 16 towns capped at 50 from TRAINING only
    """
    tr, va = tr.copy(), va.copy()
    if drop_censored:
        tr = tr[tr[TARGET] < 50]                 # training only: validation keeps them
    Xtr, Xva = tr[FEATURES], va[FEATURES]
    if scaler is not None:
        Xtr = scaler.fit_transform(Xtr)          # statistics from training rows only
        Xva = scaler.transform(Xva)
    y = np.log(tr[TARGET]) if log_target else tr[TARGET]
    fitted = LinearRegression().fit(Xtr, y)
    pred = fitted.predict(Xva)
    if log_target:
        pred = np.exp(pred)                      # back to dollars before scoring
    return rmse(pred, va[TARGET])

print(f"reproduces the baseline: {run_experiment(train, val):.3f}")

reproduces the baseline: 4.716


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("matches the baseline exactly", lambda: abs(run_experiment(train, val) - baseline_rmse) < 1e-9)
check("the log option changes something", lambda: run_experiment(train, val, log_target=True) != baseline_rmse)
check("caller's data untouched", lambda: int((train[TARGET] == 50).sum()) > 0)

  matches the baseline exactly: PASS
  the log option changes something: PASS
  caller's data untouched: PASS


True

In [ ]:
runs = {
    "0 baseline":        dict(),
    "1 standard scaler": dict(scaler=StandardScaler()),
    "2 log(price)":      dict(log_target=True),
    "3 drop censored":   dict(drop_censored=True),
}
logbook = pd.DataFrame([{"run": name, "val_rmse": round(run_experiment(train, val, **cfg), 4)}
                        for name, cfg in runs.items()])
logbook["delta_vs_baseline"] = (logbook.val_rmse - logbook.val_rmse[0]).round(4)
logbook

,run,val_rmse,delta_vs_baseline
0,0 baseline,4.7157,0.0000
1,1 standard scaler,4.7157,0.0000
2,2 log(price),4.3155,-0.4002
3,3 drop censored,4.8100,0.0943


Three results, three different kinds of lesson:

- **The scaler changes the RMSE by `0.0000`** — not "a little", *exactly* nothing. Ordinary least
  squares has a closed-form solution that is invariant to rescaling the features: multiply a column
  by 1000 and its coefficient divides by 1000, leaving every prediction identical. Scaling is not
  useless in general — it matters the moment you fit by gradient descent, or add a penalty (Ridge,
  Lasso), or use a distance (KNN, SVM). It is useless **here**, and that is worth writing down so
  you never test it again on a plain linear regression.
- **log(price) helps**, by about 0.37. Prices are right-skewed; the log compresses the expensive
  tail so a handful of costly towns stop dominating the fit.
- **Dropping the censored towns from training makes it *worse*.** Counterintuitive, and worth a
  minute: those 16 towns are the model's only examples of expensive places. Remove them and the
  model under-predicts the top of the market — while validation still contains them.

## 4. Now the uncomfortable part

You just measured three deltas on **one** split of 506 rows. How much does the answer depend on
which towns happened to land in validation?

### Exercise 5 — re-run the whole campaign on 30 different splits

In [ ]:
rows = []
for seed in range(30):
    tr, va, _ = split(boston, seed=seed)
    scores = {name: run_experiment(tr, va, **cfg) for name, cfg in runs.items()}
    rows.append(scores)
raw = pd.DataFrame(rows)

results = pd.DataFrame({
    "mean_rmse": raw.mean().round(3),
    "std_over_splits": raw.std().round(3),
    "paired_delta": (raw.sub(raw["0 baseline"], axis=0)).mean().round(3),
    "delta_std": (raw.sub(raw["0 baseline"], axis=0)).std().round(3),
    "wins_of_30": (raw.sub(raw["0 baseline"], axis=0) < 0).sum(),
})
print(results.to_string())
print(f"\nthe SPLIT alone moves the baseline by +/- {raw['0 baseline'].std():.3f} RMSE")
results

                   mean_rmse  std_over_splits  paired_delta  delta_std  wins_of_30
0 baseline             4.832            0.675         0.000      0.000           0
1 standard scaler      4.832            0.675         0.000      0.000          15
2 log(price)           4.463            0.710        -0.369      0.174          30
3 drop censored        5.004            0.789         0.173      0.268           8

the SPLIT alone moves the baseline by +/- 0.675 RMSE


,mean_rmse,std_over_splits,paired_delta,delta_std,wins_of_30
0 baseline,4.832,0.675,0.000,0.000,0
1 standard scaler,4.832,0.675,0.000,0.000,15
2 log(price),4.463,0.710,-0.369,0.174,30
3 drop censored,5.004,0.789,0.173,0.268,8


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("30 splits measured", lambda: results is not None and "wins_of_30" in results.columns)
check("log(price) wins on every split", lambda: results.loc["2 log(price)", "wins_of_30"] == 30)
check("the scaler never changes anything", lambda: abs(results.loc["1 standard scaler", "paired_delta"]) < 1e-9)

  30 splits measured: PASS
  log(price) wins on every split: PASS
  the scaler never changes anything: PASS


True

Read the two columns that matter:

| | paired delta | wins |
|---|---|---|
| standard scaler | **0.000 ± 0.000** | — |
| log(price) | **−0.369 ± 0.171** | **30 / 30** |
| drop censored | +0.173 ± 0.264 | 8 / 30 |

And then the line underneath: **the split alone moves the baseline by ±0.664.**

That is nearly **twice** the best real effect you measured. Which means a single-split comparison
cannot resolve any of these changes — not log(price), not the scaler, not outlier removal. On one
split you could see log(price) "lose" and conclude the wrong thing, or see a useless change "win"
by half a point and keep it forever.

The fix is cheap and it is the whole method:

- Compare **paired** — the same splits for both configurations, so the split cancels out.
- Repeat over ~20–30 seeds, and look at **how often** a change wins, not just the average.
- "Better on 30/30 splits" is a claim you can defend. "Better by 0.4 on my split" is not.

**The punchline:** with 506 rows, your experiment design matters more than your model choice.

## 5. The final measurement

Decisions made. Now, exactly once: refit the winner on train **+** validation, and score the test set.

In [ ]:
winner = results.paired_delta.idxmin()
full_train = pd.concat([train, val], ignore_index=True)
test_rmse = run_experiment(full_train, test, **runs[winner])

print(f"winner             : {winner}")
print(f"its validation RMSE: {results.loc[winner, 'mean_rmse']:.3f}  (mean over 30 splits)")
print(f"its TEST RMSE      : {test_rmse:.3f}   <- measured once, never used to choose anything")
print(f"dumb baseline      : {dumb_rmse:.3f}")

winner             : 2 log(price)
its validation RMSE: 4.463  (mean over 30 splits)
its TEST RMSE      : 4.698   <- measured once, never used to choose anything
dumb baseline      : 9.712


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("the test set was scored", lambda: test_rmse is not None)
check("log(price) is the winner", lambda: results.paired_delta.idxmin() == "2 log(price)")
check("still beats the dumb baseline comfortably", lambda: test_rmse < dumb_rmse * 0.7)

  the test set was scored: PASS
  log(price) is the winner: PASS
  still beats the dumb baseline comfortably: PASS


True

## What to take away

1. **Report the dumb baseline next to your model.** 4.83 against 9.22 is a result; "4.83" alone is
   a number.
2. **One change per run**, everything else frozen, seed written down.
3. **The scaler does exactly nothing to a plain linear regression** — `0.0000`, by construction,
   not by luck. Stop testing it; start testing it again the day you add a penalty or a gradient.
4. **log(price) is a real gain here** — 30/30 splits — because prices are skewed.
5. **Removing inconvenient rows from training can hurt**, when validation still contains them.
6. **On 506 rows the split is worth ±0.66 RMSE**, bigger than any change you are testing. Compare
   paired, over many seeds, and count wins.
7. **The test set is spent once**, at the end.

### Take it home
Run the same campaign with two more ideas of your own — a `RobustScaler`, or dropping `B` — and
bring the table. The interesting column is `wins_of_30`, not the mean.